In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

class BaggingClassifier:
    def __init__(self, base_classifier, n_estimators):
        self.base_classifier = base_classifier
        self.n_estimators = n_estimators
        self.classifiers = []  # stores trained classifiers

    def fit(self, X, Y):
        """Train n_estimators classifiers, each on a bootstrap sample."""
        self.classifiers = []  # reset on each fit call
        for _ in range(self.n_estimators):
            # Bootstrap sampling: randomly pick len(X) indices WITH replacement.
            # Some samples appear multiple times, others are left out (~37%).
            # This diversity is what makes bagging work.
            indices = np.random.choice(len(X), len(X), replace=True)
            X_sampled, Y_sampled = X[indices], Y[indices]

            # Create a fresh copy of the base classifier and train it
            clf = self.base_classifier.__class__()
            clf.fit(X_sampled, Y_sampled)
            self.classifiers.append(clf)

        return self

    def predict(self, X):
        """Predict by majority voting across all trained classifiers."""
        # Each classifier predicts for all samples → shape: (n_estimators, n_samples)
        predictions = np.array([clf.predict(X) for clf in self.classifiers])

        # np.apply_along_axis(func, axis, arr):
        #   Applies a function along a specific axis of an array.
        #   axis=0 means "go down the rows" → for each column (sample),
        #   collect all n_estimators predictions and pass them to func.
        #
        # np.bincount(x):
        #   Counts occurrences of each non-negative integer in x.
        #   Example: bincount([0, 1, 1, 2, 1]) → [1, 3, 1]
        #            (0 appears 1x, 1 appears 3x, 2 appears 1x)
        #
        # .argmax() returns the index of the largest count → the most voted class.
        #
        # Together: for each sample, count the votes from all classifiers,
        # then pick the class with the most votes.
        majority_votes = np.apply_along_axis(
            lambda x: np.bincount(x).argmax(), axis=0, arr=predictions
        )
        return majority_votes


# Load the digits dataset (8x8 images of handwritten digits 0-9)
digits = load_digits()
X, Y = digits.data, digits.target
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# Train Bagging Classifier (10 decision trees on bootstrap samples)
base_clf = DecisionTreeClassifier()
model = BaggingClassifier(base_classifier=base_clf, n_estimators=10)
model.fit(X_train, Y_train)

# Evaluate ensemble accuracy (majority vote of all 10 trees)
Y_pred = model.predict(X_test)
print(f"Bagging Ensemble Accuracy: {accuracy_score(Y_test, Y_pred):.2f}")

# Compare with each individual tree's accuracy
print("\nIndividual classifier accuracies:")
for i, clf in enumerate(model.classifiers):
    acc = accuracy_score(Y_test, clf.predict(X_test))
    print(f"  Tree {i+1}: {acc:.2f}")


Bagging Ensemble Accuracy: 0.94

Individual classifier accuracies:
  Tree 1: 0.81
  Tree 2: 0.85
  Tree 3: 0.84
  Tree 4: 0.83
  Tree 5: 0.85
  Tree 6: 0.82
  Tree 7: 0.86
  Tree 8: 0.88
  Tree 9: 0.82
  Tree 10: 0.82
